# Data exploration: transactions and fraud alerts

**Goal:** understand the data before building any rule or model. We work with three tables covering 5 older adults and their bank activity, and we look for patterns of possible fraud or financial exploitation.

**How the notebook is organized:**
1. Loading and reviewing the three tables.
2. First charts: how much money moves, how many transactions there are, and which ones are labeled.
3. Derived variables to measure how "unusual" an amount is for each user.
4. Money in and money out for each user.
5. Additional exploration (at the end of the notebook) to look for specific patterns.

All charts are built with **Altair**.

## 1. Loading the data

**What I'm doing:** reading the three CSV files with pandas.

**Why:** each file contributes a different piece of the problem:
- `transactions`: the movements (amount, date, type, category, channel).
- `ground_truth_labels`: which transactions are flagged as suspicious and why.
- `user_profiles`: who each user is (age, monthly income, city).

They can be joined with `transaction_id` (transactions ↔ labels) and `user_id` (transactions ↔ profiles).

In [3]:
import pandas as pd
df_ground_truth_labels = pd.read_csv('../data/ground_truth_labels.csv')
df_transactions = pd.read_csv('../data/transactions.csv')
df_user_profiles = pd.read_csv('../data/user_profiles.csv')

## 2. Reviewing the labels

**What I'm doing:** looking at summary statistics (`describe`) and the last rows (`tail`) of the labels table.

**Why:** before plotting anything I want to know how many classes there are and how imbalanced they are. The summary shows that almost every transaction is `benign` (259 out of 291), so alerts are very rare. This matters because any rule or model we build later will have few positive examples to learn from or to be validated on.

`tail()` confirms the format: `alert_type` and `notes` are only filled in for flagged rows.

In [4]:
df_ground_truth_labels.describe()

,transaction_id,ground_truth_label,alert_type,notes
count,291,291,32,32
unique,291,3,5,5
top,TXN-770487,benign,unusual_spending,Spending completely inconsistent with user pro...
freq,1,259,12,12


In [5]:
df_ground_truth_labels.tail()

,transaction_id,ground_truth_label,alert_type,notes
286,TXN-931621,benign,NaN,NaN
287,TXN-713799,benign,NaN,NaN
288,TXN-416282,benign,NaN,NaN
289,TXN-847006,benign,NaN,NaN
290,TXN-988456,benign,NaN,NaN


## 3. Reviewing the transactions

**What I'm doing:** a summary of `amount` and the first rows of the transactions table.

**Why:** I want to see the shape of the amounts. The mean (~613) is far above the median (~115) and the maximum reaches 6,000, so most movements are small and a few are huge. That long tail is where fraud tends to hide, but it is also where legitimate income lives (pensions, deposits).

`head()` shows the available columns: type (`credit`/`debit`), `raw_description` (merchant or recipient text), category and channel.

In [6]:
df_transactions.describe()

,amount
count,291.000000
mean,612.758625
std,1163.766570
min,15.260000
25%,55.495000
50%,114.600000
75%,385.000000
max,6000.000000


In [7]:
df_transactions.head()

,transaction_id,user_id,date,amount,type,raw_description,category,channel
0,TXN-770487,USR-001,2024-07-01 06:01:47,3200.00,credit,OHIO STRS PENSION DIRECT DEP,income,ACH
1,TXN-356787,USR-001,2024-07-04 11:08:47,50.12,debit,KROGER #1234 COLUMBUS OH,groceries,POS
2,TXN-717889,USR-001,2024-07-08 12:51:55,15.26,debit,CVS/PHARMACY #0567 COLUMBUS OH,pharmacy,POS
3,TXN-671858,USR-001,2024-07-12 09:37:27,46.59,debit,KROGER #1234 COLUMBUS OH,groceries,POS
4,TXN-733052,USR-001,2024-07-12 12:51:02,28.78,debit,TIM HORTONS #88 COLUMBUS OH,dining,POS


## 4. User profiles

**What I'm doing:** reviewing the profiles table.

**Why:** there are only 5 users (aged 68 to 81) with monthly incomes between 2,100 and 5,800. I need this context because "unusual" is relative: a $2,000 transfer is routine for someone earning $5,800 and huge for someone earning $2,100. City and account type also help me spot out-of-place spending.

In [8]:
df_user_profiles.describe()

,age,monthly_income
count,5.000000,5.000000
mean,74.800000,3680.000000
std,5.263079,1472.073368
min,68.000000,2100.000000
25%,72.000000,2800.000000
50%,74.000000,3200.000000
75%,79.000000,4500.000000
max,81.000000,5800.000000


In [9]:
df_user_profiles.head()

,user_id,name,age,occupation,monthly_income,location,account_type,bio
0,USR-001,Margaret Thompson,74,Retired Teacher,3200,"Columbus, OH",Checking + Savings,Retired elementary school teacher. Lives alone...
1,USR-002,Robert Chen,68,Retired Engineer (still consults),5800,"Scottsdale, AZ",Checking + Brokerage,Retired aerospace engineer who still does occa...
2,USR-003,Dorothy Williams,81,Retired,2100,"Tampa, FL",Checking,Retired postal worker. Very fixed routine - sa...
3,USR-004,Frank Martinez,72,Retired Police Officer,4500,"San Antonio, TX",Checking + Savings,Retired police officer with a good pension. Ge...
4,USR-005,Helen Park,79,Retired,2800,"Portland, OR",Checking,Retired librarian. Lives independently. Enjoys...


## 5. Spending categories

**What I'm doing:** counting transactions per category, including nulls (`dropna=False`).

**Why:** I want to know which categories dominate and whether there is missing data. `groceries` and `dining` are the most frequent (daily routine). `transfer` (37 transactions) interests me the most: it is how money leaves to third parties, the typical channel for exploitation and money mules.

In [10]:
df_transactions["category"].value_counts(dropna=False)

category
groceries             80
dining                50
transfer              37
income                26
shopping              19
pharmacy              16
utilities             16
travel                 9
deposit                6
financial_services     6
entertainment          5
lodging                5
charitable             4
recreation             4
personal_care          4
housing                4
Name: count, dtype: int64

## 6. Total amount per week and user

**What I'm doing:** converting `date` to a real datetime, adding each user's name with a `merge`, summing amounts per week and plotting one line per user.

**Why:** a quick view over time shows who moves the most money and whether there are spikes. I group by week rather than by day because with ~290 transactions over 4 months a daily chart would be mostly noise.

**Heads-up:** `amount` is always positive, so this sum mixes money in and money out. Weeks with a pension or deposits look high even though that is not spending. I separate both flows further down.

In [11]:
import altair as alt

# Fecha a datetime
df_transactions["date"] = pd.to_datetime(df_transactions["date"])

# Agregar nombre del usuario
df = df_transactions.merge(
    df_user_profiles[["user_id", "name"]],
    on="user_id",
    how="left"
)

# Agrupar por semana (freq="W")
df_daily = (
    df.groupby(
        ["name", pd.Grouper(key="date", freq="W")],
        as_index=False
    )["amount"]
    .sum()
)

# Graficar
chart = (
    alt.Chart(df_daily)
    .mark_line(point=True)
    .encode(
        x=alt.X("date:T", title="Fecha"),
        y=alt.Y("amount:Q", title="Monto"),
        color=alt.Color("name:N", title="Usuario"),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("date:T", title="Fecha"),
            alt.Tooltip("amount:Q", title="Monto")
        ]
    )
    .properties(
        title="Monto por fecha y usuario",
        width=900,
        height=500
    )
)

chart

alt.Chart(...)

I check the first rows of the weekly aggregate to confirm it has the expected shape: one row per user and week, with the total amount.

In [12]:
df_daily.head()

,name,date,amount
0,Dorothy Williams,2024-07-07,2153.47
1,Dorothy Williams,2024-07-14,108.35
2,Dorothy Williams,2024-07-21,221.25
3,Dorothy Williams,2024-07-28,64.96
4,Dorothy Williams,2024-08-04,2207.77


## 7. Number of transactions per week and user

**What I'm doing:** instead of summing amounts, I count how many transactions each user makes per week.

**Why:** amount and frequency tell different stories. A sudden change in the number of movements (suddenly many, or suddenly none) can also signal that something changed in the account, even if the amounts are small.

In [13]:
df = df_transactions.merge(
    df_user_profiles[["user_id", "name"]],
    on="user_id"
)

df_count = (
    df.groupby(
        [
            "name",
            pd.Grouper(key="date", freq="W")
        ]
    )
    .size()
    .reset_index(name="cantidad_transacciones")
)

chart = (
    alt.Chart(df_count)
    .mark_line(point=True)
    .encode(
        x="date:T",
        y="cantidad_transacciones:Q",
        color=alt.Color("name:N", title="Usuario"),
        tooltip=["name", "date", "cantidad_transacciones"]
    )
    .properties(
        width=900,
        height=500,
        title="Cantidad de transacciones por usuario"
    )
)

chart

alt.Chart(...)

## 8. Labeled transactions, one panel per user

**What I'm doing:** joining transactions, labels and names, and plotting each transaction as a point (date vs. amount) colored by its label. Each user gets their own panel, with the name shown on the left.

**Why:** with all users in one chart the points overlapped and it was hard to see what happened to each person. Splitting by user makes the pattern of each case visible. I use an independent vertical scale per panel because amounts differ a lot between users; this shows the shape of each series (the trade-off is that heights cannot be compared across panels).

In [14]:
# Join: transacciones + etiquetas + nombre del usuario
df = (
    df_transactions
    .merge(
        df_ground_truth_labels,
        on="transaction_id",
        how="left"
    )
    .merge(
        df_user_profiles[["user_id", "name"]],
        on="user_id",
        how="left"
    )
)

# Gráfico base: un punto por transacción, coloreado por etiqueta
puntos = (
    alt.Chart(df)
    .mark_circle(size=120)
    .encode(
        x=alt.X("date:T", title="Fecha"),
        y=alt.Y("amount:Q", title="Monto"),
        color=alt.Color(
            "ground_truth_label:N",
            title="Etiqueta",
            scale=alt.Scale(
                domain=["benign", "suspicious", "alert"],
                range=["#4CAF50", "#FF9800", "#F44336"]
            )
        ),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("transaction_id:N", title="Transacción"),
            alt.Tooltip("date:T", title="Fecha"),
            alt.Tooltip("amount:Q", title="Monto"),
            alt.Tooltip("category:N", title="Categoría"),
            alt.Tooltip("ground_truth_label:N", title="Label"),
            alt.Tooltip("alert_type:N", title="Tipo Alerta"),
            alt.Tooltip("notes:N", title="Notas")
        ]
    )
    .properties(width=1000, height=150)
)

# Un panel por usuario, con el nombre como encabezado
chart = (
    puntos
    .facet(
        row=alt.Row(
            "name:N",
            title="Usuario",
            sort=df_user_profiles["name"].tolist(),
            header=alt.Header(labelAngle=0, labelAlign="left")
        )
    )
    .resolve_scale(y="independent")
    .properties(title="Transacciones etiquetadas por usuario")
)

chart

alt.FacetChart(...)

## 9. Variables to measure how "unusual" an amount is

**What I'm doing:** computing each user's mean and standard deviation of amounts, and using them to create several new variables:
- `zscore`: how many standard deviations an amount is from what is usual for that user.
- `pct_income`: the amount as a percentage of the user's monthly income.
- `flow`: `Money In` for a credit, `Money Out` for a debit.
- `df_spending`: I keep only the **outgoing `transfer` debits** and compute `relative_to_mean` (amount ÷ the user's mean).

**Why:** an absolute amount says nothing by itself; what matters is how far it is from normal *for that person*. I focus on outgoing transfers because that is where exploitation and money mules show up.

**Limitation:** the mean and standard deviation are computed over *all* of a user's transactions (credits and debits), so pensions and deposits inflate them. It is an approximate reference, not an exact spending baseline.

In [15]:
import numpy as np

In [16]:
user_stats = (
    df_transactions
    .groupby("user_id")["amount"]
    .agg(["mean", "std"])
    .reset_index()
)

df = (
    df_transactions
    .merge(user_stats, on="user_id")
    .merge(
        df_user_profiles[["user_id", "monthly_income", "name"]],
        on="user_id",
        how="left"
    )
)

df["zscore"] = (df["amount"] - df["mean"]) / df["std"]

df["pct_income"] = (
    df["amount"] /
    df["monthly_income"] * 100
)

df["flow"] = np.where(
    df["type"] == "credit",
    "Money In",
    "Money Out"
)

df_spending = df[df["type"] == "debit"].copy()
df_spending = df_spending[
    df_spending["category"] == "transfer"
]
df_spending["relative_to_mean"] = (
    df_spending["amount"] / df_spending["mean"]
)

## 10. Outgoing transfers vs. each user's mean

**What I'm doing:** one panel per user with each outgoing transfer as a point (red if it is above the user's mean, blue if not) and a green line showing that mean.

**Why:** it is a visual way of asking "which transfers stand out for this person?". If red points cluster together and grow over time, it is a warning sign (for example, transfers escalating toward the same recipient).

In [17]:
points = (
    alt.Chart(df_spending)
    .mark_circle(size=100)
    .encode(
        x=alt.X("date:T", title="Fecha"),
        y=alt.Y("amount:Q", title="Monto"),
        color=alt.condition(
            alt.datum.relative_to_mean > 1,
            alt.value("crimson"),
            alt.value("steelblue")
        ),
        tooltip=[
            alt.Tooltip("date:T", title="Fecha"),
            alt.Tooltip("transaction_id:N", title="Transaction ID"),
            alt.Tooltip("user_id:N", title="Usuario"),
            alt.Tooltip("amount:Q", title="Monto", format=",.2f"),
            alt.Tooltip("zscore:Q", title="Z-Score", format=".2f"),
            alt.Tooltip("category:N", title="Categoría"),
            alt.Tooltip("channel:N", title="Canal"),
            alt.Tooltip("raw_description:N", title="Descripción")
        ]
    )
)

mean_line = (
    alt.Chart(df_spending)
    .mark_rule(
        color="green",
        size=2
    )
    .encode(
        y=alt.Y("mean:Q")
    )
)

chart = (
    points +
    mean_line
).properties(
    width=900,
    height=180,
)

chart.facet(
    row=alt.Row("name:N", title="Usuario")
).resolve_scale(
    y="independent"
).properties(
    title="Gastos anómalos por usuario"
)

alt.FacetChart(...)

## 11. Money in and money out, one panel per user with their name

**What I'm doing:** creating `signed_amount`: positive when money comes in (credit) and negative when it goes out (debit). Before that I add each user's name to the table.

**Why:** this lets me see both flows on a single axis. Green = in, red = out. It is the key view for spotting **money mules**: accounts where a large amount comes in and almost the same amount goes out shortly after. Each panel now shows the user's name instead of their `user_id`, so it can be read without looking up the profiles table.

In [18]:
# Agrego el nombre del usuario para poder mostrarlo en los gráficos
df_mule = df_transactions.merge(
    df_user_profiles[["user_id", "name"]],
    on="user_id",
    how="left"
)

# Entra (+) / sale (-)
df_mule["signed_amount"] = np.where(
    df_mule["type"] == "credit",
    df_mule["amount"],
    -df_mule["amount"]
)

In [19]:
alt.Chart(df_mule).mark_bar().encode(
    x=alt.X("date:T", title="Fecha"),
    y=alt.Y("signed_amount:Q", title="Monto"),
    color=alt.condition(
        alt.datum.signed_amount > 0,
        alt.value("forestgreen"),
        alt.value("crimson")
    ),
    tooltip=[
        alt.Tooltip("name:N", title="Usuario"),
        "date:T",
        "amount:Q",
        "type:N",
        "category:N",
        "raw_description:N"
    ]
).properties(
    width=900,
    height=250
).facet(
    row=alt.Row(
        "name:N",
        title="Usuario",
        sort=df_user_profiles["name"].tolist(),
        header=alt.Header(labelAngle=0, labelAlign="left")
    )
)

alt.FacetChart(...)

## 12. Automatic report (optional, did not work in this environment)

I tried to generate an automatic report with `ydata_profiling`. It failed because `pkg_resources` was missing, and the attempts to install packages with `pip` did not fix it. I leave the cells as a record: they are **not needed** for the rest of the analysis, which is done directly with pandas and Altair.

In [20]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_transactions)
profile.to_file("reporte.html")

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
pip install ydata_profiling

Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 4.0.0 Requires-Python >=3.7,<3.11; 4.1.0 Requires-Python >=3.7,<3.12; 4.1.1 Requires-Python >=3.7,<3.12; 4.1.2 Requires-Python >=3.7,<3.12; 4.10.0 Requires-Python >=3.7,<3.13; 4.11.0 Requires-Python >=3.7,<3.13; 4.12.0 Requires-Python >=3.7,<3.13; 4.12.1 Requires-Python >=3.7,<3.13; 4.12.2 Requires-Python >=3.7,<3.13; 4.13.0 Requires-Python >=3.7,<3.13; 4.14.0 Requires-Python >=3.7,<3.13; 4.15.0 Requires-Python >=3.7,<3.13; 4.15.1 Requires-Python >=3.7,<3.13; 4.16.0 Requires-Python >=3.7,<3.13; 4.16.1 Requires-Python >=3.7,<3.13; 4.17.0 Requires-Python >=3.7,<3.14; 4.18.0 Requires-Python >=3.10,<3.14; 4.18.1 Requires-Python >=3.10,<3.14; 4.18.4 Requires-Python >=3.10,<3.14; 4.2.0 Requires-Python >=3.7,<3.12; 4.3.0 Requires-Python >=3.7,<3.12; 4.3.1 Requires-Python >=3.7,<3.12; 4.3.2 Requires-Python >=3.7,<3.12; 4.4.0 Requires-Python >=3.7,<3.12; 4.5.0 Requires-Python >=3.7,<3.12; 4.5.1 Requires-Python >=3.7,

In [ ]:
pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 9.8/9.8 MB 68.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ------------------- -------------------- 6.3/12.7 MB 32.4 MB/s eta 0:00:01
   -------------------------------------- - 12.3/12.7 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 26.0 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


---
# Additional exploration

So far we have seen the big picture. Now I look for more specific patterns: which alerts exist and how much money they involve, whether money comes in and goes out in nearly equal amounts (mules), which categories suddenly appear, how large amounts are relative to each user's income, and which days stand out against a 7-day rolling baseline.

## 13. Unified base table

**What I'm doing:** joining transactions, labels and profiles into a single DataFrame (`df_full`) and creating derived columns (`flujo`, `signed_amount`, `mes`, `pct_income`, `is_flagged`). At the end I build a per-user summary.

**Why:** in the earlier cells the variable `df` kept being overwritten with different contents. A new, complete table avoids depending on execution order and lets me plot any combination without repeating merges.

In [21]:
# Tabla unificada: transacciones + etiquetas + perfil del usuario
df_full = (
    df_transactions
    .merge(df_ground_truth_labels, on="transaction_id", how="left")
    .merge(
        df_user_profiles[["user_id", "name", "age", "monthly_income", "location"]],
        on="user_id",
        how="left"
    )
)

# Variables derivadas
df_full["date"] = pd.to_datetime(df_full["date"])
df_full["flujo"] = np.where(df_full["type"] == "credit", "Entra", "Sale")
df_full["signed_amount"] = np.where(df_full["type"] == "credit", df_full["amount"], -df_full["amount"])
df_full["mes"] = df_full["date"].dt.strftime("%Y-%m")
df_full["pct_income"] = df_full["amount"] / df_full["monthly_income"] * 100
df_full["is_flagged"] = df_full["ground_truth_label"] != "benign"

# Orden de usuarios y paleta de etiquetas para todos los gráficos nuevos
USUARIOS = df_user_profiles["name"].tolist()
COLOR_LABEL = alt.Scale(
    domain=["benign", "suspicious", "alert"],
    range=["#4CAF50", "#FF9800", "#F44336"]
)

# Resumen por usuario: transacciones, dinero que entra/sale y cuántas están marcadas
resumen = df_full.pivot_table(
    index="name", columns="flujo", values="amount", aggfunc="sum", fill_value=0
)
resumen["transacciones"] = df_full.groupby("name").size()
resumen["marcadas"] = df_full.groupby("name")["is_flagged"].sum()
resumen.loc[USUARIOS]

flujo,Entra,Sale,transacciones,marcadas
name,,,,
Margaret Thompson,12800.00,9687.33,55,8
Robert Chen,18617.76,27002.04,70,0
Dorothy Williams,35400.00,27550.80,48,12
Frank Martinez,18000.00,7749.98,59,0
Helen Park,11200.00,10304.85,59,12


## 14. Which alerts exist and how much money do they involve?

**What I'm doing:** summing the amount of flagged transactions (`suspicious` or `alert`) per user and splitting them by alert type.

**Why:** counting alerts is not enough; I want to know how much money is at stake and what kind of problem each person has. I fix the axis to all 5 users so it is clear who has **no** flags at all.

In [22]:
df_alertas = df_full[df_full["is_flagged"]]

chart = (
    alt.Chart(df_alertas)
    .mark_bar()
    .encode(
        x=alt.X(
            "name:N",
            title="Usuario",
            scale=alt.Scale(domain=USUARIOS),
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y("sum(amount):Q", title="Monto total marcado"),
        color=alt.Color("alert_type:N", title="Tipo de alerta"),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("alert_type:N", title="Tipo de alerta"),
            alt.Tooltip("count():Q", title="Transacciones"),
            alt.Tooltip("sum(amount):Q", title="Monto", format=",.2f")
        ]
    )
    .properties(
        width=700,
        height=350,
        title="Monto marcado como suspicious/alert por usuario y tipo de alerta"
    )
)

chart

alt.Chart(...)

## 15. Total money in vs. total money out, per user

**What I'm doing:** grouped bars with the total money coming in (green) and going out (red) for each user.

**Why:** it is a quick sanity check on proportions. In a normal account, inflows are few and regular (pension) and outflows are many and small. If a user moves much more money than their income, or if inflows and outflows are similar, they deserve a closer look. Note Dorothy: she moves a lot of money for a monthly income of only 2,100.

In [23]:
chart = (
    alt.Chart(df_full)
    .mark_bar()
    .encode(
        x=alt.X(
            "name:N",
            title="Usuario",
            scale=alt.Scale(domain=USUARIOS),
            axis=alt.Axis(labelAngle=0)
        ),
        xOffset=alt.XOffset("flujo:N"),
        y=alt.Y("sum(amount):Q", title="Monto total"),
        color=alt.Color(
            "flujo:N",
            title="Flujo",
            scale=alt.Scale(domain=["Entra", "Sale"], range=["forestgreen", "crimson"])
        ),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("flujo:N", title="Flujo"),
            alt.Tooltip("count():Q", title="Transacciones"),
            alt.Tooltip("sum(amount):Q", title="Monto total", format=",.2f")
        ]
    )
    .properties(width=700, height=350, title="Dinero que entra y sale por usuario")
)

chart

alt.Chart(...)

## 16. Cumulative net flow over time

**What I'm doing:** sorting transactions by date and accumulating `signed_amount` for each user. Flagged transactions are marked with red dots.

**Why:** it shows how the "balance" evolves over time and when the alerts happen. A user with a stable routine goes up with each pension and down slowly with expenses; sudden jumps or trend changes deserve a review.

**Heads-up:** the data does not include the account balance, so this is the net flow *since the start of the period*, not the real balance.

In [24]:
df_flujo = df_full.sort_values(["user_id", "date"]).copy()
df_flujo["flujo_acumulado"] = df_flujo.groupby("user_id")["signed_amount"].cumsum()

linea = (
    alt.Chart(df_flujo)
    .mark_line(interpolate="step-after", color="steelblue")
    .encode(
        x=alt.X("date:T", title="Fecha"),
        y=alt.Y("flujo_acumulado:Q", title="Flujo neto acumulado")
    )
)

marcadas = (
    alt.Chart(df_flujo)
    .mark_circle(color="crimson", size=70)
    .transform_filter("datum.is_flagged")
    .encode(
        x="date:T",
        y="flujo_acumulado:Q",
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("date:T", title="Fecha"),
            alt.Tooltip("amount:Q", title="Monto", format=",.2f"),
            alt.Tooltip("type:N", title="Tipo"),
            alt.Tooltip("alert_type:N", title="Tipo de alerta"),
            alt.Tooltip("raw_description:N", title="Descripción")
        ]
    )
)

chart = (
    (linea + marcadas)
    .properties(width=900, height=150)
    .facet(
        row=alt.Row(
            "name:N",
            title="Usuario",
            sort=USUARIOS,
            header=alt.Header(labelAngle=0, labelAlign="left")
        )
    )
    .properties(title="Flujo neto acumulado por usuario (puntos rojos = transacciones marcadas)")
)

chart

alt.FacetChart(...)

## 17. Does money come in and almost the same amount go out soon after? (mule pattern)

**What I'm doing:** for each credit I look for the **first outgoing transfer by the same user within the next 72 hours** (`merge_asof`) and compute what fraction of the amount goes out. If between 85% and 100% leaves, I flag it as a possible *pass-through*.

**Why:** in a money mule the person receives funds from a third party and forwards almost all of them, keeping a small cut. The pattern is hard to see when looking at isolated transactions, but clear once they are paired.

In the chart, points on the dashed diagonal mean "exactly what came in went out". Red points close to the diagonal are the candidates.

**Heads-up:** `merge_asof` takes the *first* later outflow, so some gray pairs are not real (for example, a pension followed by an outflow that actually belongs to another deposit). That is why I filter on the ratio of amount out to amount in.

In [25]:
# Entradas (créditos) y salidas (transferencias salientes), ordenadas por fecha
creditos = (
    df_full[df_full["type"] == "credit"]
    [["user_id", "name", "date", "amount", "raw_description", "ground_truth_label"]]
    .rename(columns={
        "date": "fecha_entrada",
        "amount": "monto_entrada",
        "raw_description": "origen",
        "ground_truth_label": "label_entrada"
    })
    .sort_values("fecha_entrada")
)

transferencias = (
    df_full[(df_full["type"] == "debit") & (df_full["category"] == "transfer")]
    [["user_id", "date", "amount", "raw_description", "ground_truth_label"]]
    .rename(columns={
        "date": "fecha_salida",
        "amount": "monto_salida",
        "raw_description": "destino",
        "ground_truth_label": "label_salida"
    })
    .sort_values("fecha_salida")
)
# merge_asof usa la columna clave y no la conserva, así que guardo una copia de la fecha de salida
transferencias["fecha_salida_original"] = transferencias["fecha_salida"]

# Para cada entrada, la primera salida del mismo usuario dentro de 72 h
pares = pd.merge_asof(
    creditos,
    transferencias,
    left_on="fecha_entrada",
    right_on="fecha_salida",
    by="user_id",
    direction="forward",
    tolerance=pd.Timedelta("72h")
).dropna(subset=["monto_salida"])

pares["ratio"] = pares["monto_salida"] / pares["monto_entrada"]
pares["horas_hasta_salida"] = (
    (pares["fecha_salida_original"] - pares["fecha_entrada"]).dt.total_seconds() / 3600
)

# Regla: sale entre el 85% y el 100% de lo que entró
UMBRAL_RATIO = 0.85
pares["patron"] = np.where(
    pares["ratio"].between(UMBRAL_RATIO, 1.0),
    "Posible pass-through",
    "Normal"
)

# Gráfico: monto que entra vs. monto que sale, con la diagonal y = x
pares_plot = pares.drop(columns=["fecha_salida", "fecha_salida_original"])
diagonal = pd.DataFrame({"x": [0, 6500], "y": [0, 6500]})

linea_diagonal = (
    alt.Chart(diagonal)
    .mark_line(strokeDash=[4, 4], color="gray")
    .encode(x="x:Q", y="y:Q")
)

puntos = (
    alt.Chart(pares_plot)
    .mark_circle(size=130)
    .encode(
        x=alt.X("monto_entrada:Q", title="Monto que entra"),
        y=alt.Y("monto_salida:Q", title="Monto que sale (dentro de 72 h)"),
        color=alt.Color(
            "patron:N",
            title="Patrón",
            scale=alt.Scale(domain=["Posible pass-through", "Normal"], range=["crimson", "gray"])
        ),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("fecha_entrada:T", title="Fecha entrada"),
            alt.Tooltip("origen:N", title="Origen"),
            alt.Tooltip("destino:N", title="Destino"),
            alt.Tooltip("monto_entrada:Q", title="Entra", format=",.2f"),
            alt.Tooltip("monto_salida:Q", title="Sale", format=",.2f"),
            alt.Tooltip("ratio:Q", title="Sale / Entra", format=".0%"),
            alt.Tooltip("horas_hasta_salida:Q", title="Horas hasta la salida", format=".1f")
        ]
    )
)

chart = (
    (linea_diagonal + puntos)
    .properties(width=600, height=450, title="Entrada vs. salida posterior (la diagonal = sale todo lo que entró)")
)

# Tabla con los pares detectados
display(
    pares[pares["patron"] == "Posible pass-through"]
    [["name", "fecha_entrada", "monto_entrada", "monto_salida", "ratio", "horas_hasta_salida", "label_entrada", "label_salida"]]
    .reset_index(drop=True)
)

chart

,name,fecha_entrada,monto_entrada,monto_salida,ratio,horas_hasta_salida,label_entrada,label_salida
0,Dorothy Williams,2024-09-22 10:23:11,3500.0,3300.0,0.942857,4.373056,alert,alert
1,Dorothy Williams,2024-09-28 11:02:03,4200.0,4000.0,0.952381,29.701389,alert,alert
2,Dorothy Williams,2024-10-03 12:06:55,5000.0,4750.0,0.950000,6.193056,alert,alert
3,Dorothy Williams,2024-10-08 12:21:11,3800.0,3600.0,0.947368,28.571944,alert,alert
4,Dorothy Williams,2024-10-15 09:25:31,6000.0,5700.0,0.950000,9.253333,alert,alert
5,Dorothy Williams,2024-10-21 10:09:51,4500.0,4250.0,0.944444,29.923333,alert,alert


alt.LayerChart(...)

## 18. Which categories suddenly appear? Spending by category and month

**What I'm doing:** a heatmap per user: horizontal axis = month, vertical axis = category, color = total spending (debits only). Empty cells mean the user did not spend in that category that month.

**Why:** a change in behavior shows up as a category that used to be empty and suddenly lights up. This is much more informative than looking at isolated amounts. It also helps avoid the opposite mistake: some users (like Robert) have recurring high spending and trips, and that is normal *for them*.

In [26]:
df_gasto = df_full[df_full["type"] == "debit"]

chart = (
    alt.Chart(df_gasto)
    .mark_rect()
    .encode(
        x=alt.X("mes:O", title="Mes", axis=alt.Axis(labelAngle=-90)),
        y=alt.Y("category:N", title="Categoría"),
        color=alt.Color(
            "sum(amount):Q",
            title="Gasto total",
            scale=alt.Scale(type="sqrt", scheme="blues")
        ),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("mes:O", title="Mes"),
            alt.Tooltip("category:N", title="Categoría"),
            alt.Tooltip("sum(amount):Q", title="Gasto", format=",.2f"),
            alt.Tooltip("count():Q", title="Transacciones")
        ]
    )
    .properties(width=90, height=320)
    .facet(
        column=alt.Column(
            "name:N",
            title="Usuario",
            sort=USUARIOS
        )
    )
    .properties(title="Gasto por categoría y mes, por usuario")
)

chart

alt.FacetChart(...)

## 19. Size of spending relative to income, by label

**What I'm doing:** for each debit I compute what percentage of the user's monthly income it represents and plot it by label (`benign`, `suspicious`, `alert`) on a logarithmic scale. Each point is a transaction; I shift the points slightly sideways (jitter) so they do not hide each other. The dashed line marks 100% of monthly income.

**Why:** if flagged transactions are systematically larger *relative to the person making them*, `pct_income` is a good variable for a rule or a model. I use a log scale because values range from under 1% to over 250%.

In [27]:
puntos = (
    alt.Chart(df_gasto)
    .mark_circle(size=70, opacity=0.6)
    .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")
    .encode(
        x=alt.X(
            "ground_truth_label:N",
            title="Etiqueta",
            sort=["benign", "suspicious", "alert"],
            axis=alt.Axis(labelAngle=0)
        ),
        xOffset="jitter:Q",
        y=alt.Y(
            "pct_income:Q",
            title="Monto / ingreso mensual (%)",
            scale=alt.Scale(type="log")
        ),
        color=alt.Color("ground_truth_label:N", scale=COLOR_LABEL, legend=None),
        tooltip=[
            alt.Tooltip("name:N", title="Usuario"),
            alt.Tooltip("date:T", title="Fecha"),
            alt.Tooltip("amount:Q", title="Monto", format=",.2f"),
            alt.Tooltip("pct_income:Q", title="% del ingreso", format=".1f"),
            alt.Tooltip("category:N", title="Categoría"),
            alt.Tooltip("raw_description:N", title="Descripción"),
            alt.Tooltip("alert_type:N", title="Tipo de alerta")
        ]
    )
)

regla_100 = (
    alt.Chart(pd.DataFrame({"y": [100]}))
    .mark_rule(strokeDash=[4, 4], color="gray")
    .encode(y=alt.Y("y:Q", scale=alt.Scale(type="log")))
)

chart = (
    (puntos + regla_100)
    .properties(width=600, height=400, title="Gasto como % del ingreso mensual, por etiqueta")
)

chart

alt.LayerChart(...)

## 20. Days above the 7-day rolling baseline (average and total spending)

**What I'm doing:** for each user I build a daily series of spending (debits only) with two metrics: the **total** spent that day and the **average amount per transaction** that day. Then I compare each day against a rolling baseline built from the **7 previous days**. A day is highlighted in red only if **both** metrics are above their baseline.

**How the baselines are computed:**
- **Total baseline:** the mean daily total over the previous 7 calendar days (days with no spending count as 0).
- **Average baseline:** total spent in the previous 7 days ÷ number of transactions in those days, i.e. the average amount per transaction over the window.
- The window covers the previous 7 days and **excludes the current day** (`shift(1)`). If it included the day being evaluated, a spike would raise its own baseline and be harder to detect.
- The first 7 days of the period have no baseline yet, so they are never highlighted.

**Why:** a fixed threshold does not work across users with very different spending. A rolling baseline adapts to each person's recent behavior and asks "is this day unusual *compared to this person's last week*?". Requiring both metrics reduces noise: the daily total can rise just because of several small purchases, and the average can rise with a single purchase on a quiet day, but both rising together is a stronger signal.

**How to read the chart:** each row is one user; the left chart is the daily total and the right chart is the daily average. Blue dots are the daily values, the gray dashed line is the rolling baseline, red dots are days where both metrics are above their baseline, and black rings are days that contain a flagged outgoing transaction (`suspicious` or `alert`). The table printed above the chart counts how many alert days the rule catches.

In [30]:
# Window (in days) used for the rolling baseline
VENTANA = 7

# Spending = debits only. All users share the same calendar so the charts line up.
df_debitos = df_full[df_full["type"] == "debit"]
calendario = pd.date_range(
    df_full["date"].dt.normalize().min(),
    df_full["date"].dt.normalize().max(),
    freq="D"
)

partes = []
for user_id, g in df_debitos.groupby("user_id"):
    # Daily series: total spent, number of transactions and flagged transactions per day
    d = (
        g.assign(dia=g["date"].dt.normalize())
        .groupby("dia")
        .agg(
            gasto_total=("amount", "sum"),
            n_tx=("amount", "size"),
            alertas=("is_flagged", "sum")
        )
        .reindex(calendario, fill_value=0)  # days without spending count as 0
    )
    d.index.name = "date"

    # Average amount per transaction that day (undefined when there was no spending)
    d["gasto_promedio"] = d["gasto_total"] / d["n_tx"].replace(0, np.nan)

    # Baselines from the previous VENTANA days; shift(1) excludes the current day
    suma_total = d["gasto_total"].rolling(VENTANA, min_periods=VENTANA).sum()
    suma_tx = d["n_tx"].rolling(VENTANA, min_periods=VENTANA).sum()
    d["base_total"] = (suma_total / VENTANA).shift(1)
    d["base_promedio"] = (suma_total / suma_tx.replace(0, np.nan)).shift(1)

    # A day is highlighted only if BOTH metrics are above their baseline
    d["sup_total"] = d["gasto_total"] > d["base_total"]
    d["sup_promedio"] = d["gasto_promedio"] >  d["base_promedio"]
    d["ambos"] = d["sup_total"] & d["sup_promedio"]

    d["name"] = g["name"].iloc[0]
    partes.append(d.reset_index())

df_roll = pd.concat(partes, ignore_index=True)

# How many days are highlighted, and how many of them contain a labeled alert?
resumen_rolling = df_roll.groupby("name").agg(
    dias_con_gasto=("n_tx", lambda s: int((s > 0).sum())),
    dias_marcados_por_rolling=("ambos", "sum"),
    dias_con_alerta=("alertas", lambda s: int((s > 0).sum()))
)
resumen_rolling["alertas_capturadas"] = (
    df_roll[df_roll["ambos"] & (df_roll["alertas"] > 0)].groupby("name").size()
)
resumen_rolling = resumen_rolling.fillna(0).astype(int).loc[USUARIOS]
display(resumen_rolling)


def panel(df_u, columna, base, titulo, etiqueta_y):
    """One chart: daily value, rolling baseline, highlighted days and alert days."""
    valores = (
        alt.Chart(df_u)
        .mark_circle(color="steelblue", size=25, opacity=0.6)
        .encode(
            x=alt.X("date:T", title=None),
            y=alt.Y(f"{columna}:Q", title=etiqueta_y)
        )
    )

    baseline = (
        alt.Chart(df_u)
        .mark_line(color="gray", strokeDash=[4, 4])
        .encode(x="date:T", y=f"{base}:Q")
    )

    marcados = (
        alt.Chart(df_u)
        .mark_circle(color="crimson", size=80)
        .transform_filter("datum.ambos")
        .encode(
            x="date:T",
            y=f"{columna}:Q",
            tooltip=[
                alt.Tooltip("name:N", title="Usuario"),
                alt.Tooltip("date:T", title="Fecha"),
                alt.Tooltip("gasto_total:Q", title="Gasto total", format=",.2f"),
                alt.Tooltip("base_total:Q", title="Rolling total (7d)", format=",.2f"),
                alt.Tooltip("gasto_promedio:Q", title="Gasto promedio", format=",.2f"),
                alt.Tooltip("base_promedio:Q", title="Rolling promedio (7d)", format=",.2f"),
                alt.Tooltip("n_tx:Q", title="Transacciones"),
                alt.Tooltip("alertas:Q", title="Transacciones marcadas")
            ]
        )
    )

    con_alerta = (
        alt.Chart(df_u)
        .mark_point(color="black", size=180, filled=False, strokeWidth=1.5)
        .transform_filter("datum.alertas > 0")
        .encode(x="date:T", y=f"{columna}:Q")
    )

    return (
        (valores + baseline + marcados + con_alerta)
        .properties(width=420, height=140, title=titulo)
    )


# One row per user: total on the left, average on the right
filas = []
for nombre in USUARIOS:
    df_u = df_roll[df_roll["name"] == nombre]
    filas.append(
        alt.hconcat(
            panel(df_u, "gasto_total", "base_total", f"{nombre} - gasto total diario", "Gasto total"),
            panel(df_u, "gasto_promedio", "base_promedio", f"{nombre} - gasto promedio por transacción", "Gasto promedio")
        )
    )

chart = alt.vconcat(*filas).properties(
    title=alt.TitleParams(
        text="Días con gasto promedio y gasto total superiores al rolling de 7 días",
        subtitle=[
            "Rojo = el total y el promedio del día superan su rolling de los 7 días previos  |  Línea gris punteada = rolling",
            "Anillo negro = día con una transacción marcada (suspicious/alert)"
        ],
        anchor="start"
    )
)

chart

,dias_con_gasto,dias_marcados_por_rolling,dias_con_alerta,alertas_capturadas
name,,,,
Margaret Thompson,43,16,8,8
Robert Chen,49,20,0,0
Dorothy Williams,34,15,6,5
Frank Martinez,45,16,0,0
Helen Park,39,11,9,4


alt.VConcatChart(...)

## 21. Findings and next steps

**What the data shows**
- **Dorothy (money mule):** the 6 inflows labeled `money_mule_inflow` are matched by the rule in section 17 with an outgoing transfer of 94% to 95% of the amount, in under about 30 hours. The rule flags no other inflow from any other user.
- **Margaret (exploitation):** outgoing transfers with growing amounts to the same recipient (from $200 to $2,000, then a $1,200 wire). The recipient's name is written in several ways ("KEVIN R", "KEVIN RICHARDS"), which matters if you want to group by recipient.
- **Helen (unusual spending):** in October `lodging`, `entertainment` and `dining` suddenly appear at merchants abroad (Spain, Thailand, UAE), categories that did not exist in earlier months and do not fit her life in Portland.
- **Robert (potential false positive):** he spends a lot and travels (London, Napa), but everything is `benign`. A detector that only looks at "high amount" or "different location" would flag him by mistake.
- **Size relative to income:** among `benign` debits the median is ~2.5% of monthly income and the maximum ~40%; among those labeled `alert` the median is ~27% and it goes up to ~270%. `pct_income` separates fairly well, though it should be combined with other signals because of cases like Robert's.
- **7-day rolling baseline:** being above the baseline on both metrics is a weak signal on its own. The rule highlights 78 days in total, and 17 of them contain a labeled alert (all 8 of Margaret's alert days, 5 of Dorothy's 6 and 4 of Helen's 9). But Frank and Robert, who have no alerts, still get 16 and 20 highlighted days. It is useful as one signal to combine with others, not as a stand-alone detector.

**Limitations**
- There are only 291 transactions, 30 `alert` and 2 `suspicious`; any result should be read with caution.
- The rules in this notebook were tuned by looking at this same data, so there is no independent validation.

**Possible next steps**
1. Normalize recipient names (for example, by text similarity) to detect "new recipient with growing amounts".
2. Build a per-user baseline (median and MAD, or rolling windows) instead of the mean and standard deviation over the whole history.
3. Combine the signals (pass-through ratio, % of income, new category, out-of-pattern location, rolling-baseline flag) into a rule or score and measure how many real alerts it catches and how many false positives it generates.